# Agente de revisión de documentos de ingeniería con Gemini
Usa la API de Gemini (plan gratuito de Google AI Studio o de pago), sin necesitar GPU.

Antes de ejecutar:
1. Crea una clave en https://aistudio.google.com/apikey.
2. En Kaggle: *Add-ons → Secrets* → agrega `GEMINI_API_KEY` y actívalo para este notebook; *Settings → Internet: On*.
3. En el panel *Input* agrega el dataset con tus documentos.

⚠️ **Confidencialidad:** en el plan gratuito Google puede usar lo que envías para mejorar sus productos y revisores humanos pueden leerlo. No subas documentos confidenciales sin activar facturación o sin autorización.
⚠️ La cuota gratuita es baja (p. ej. 20 peticiones por día por modelo): si se agota, el agente cambia a otro modelo o te dice cuándo vuelve.

## Cómo usar este notebook
| Quiero… | Dónde |
|---|---|
| **Ver el informe de la revisión** | Ejecuta la celda de revisión (`review_folder`); justo debajo se muestra el informe. Los archivos `informe.md`, `hallazgos.csv` y `revision.json` quedan en el panel derecho **Output → /kaggle/working/informe** (se descargan desde ahí). |
| **Agregar, quitar o reemplazar documentos** | Celda **«Documentos a revisar»**: quita el `#` de la línea que necesites (`docs.upload()`, `docs.remove(...)`, `docs.replace(...)`). |
| **Preguntarle a la IA o pedir un resumen** | Sección **«Chat con los documentos»**: ejecuta la celda con `chat.ui()` y usa la caja de preguntas. |

**Si cambias los documentos**, vuelve a ejecutar en este orden: *Documentos a revisar → Revisión → Informe → Preparar el chat*.

In [ ]:
# Los avisos rojos 'dependency resolver ... google-colab ...' son normales en Kaggle y no impiden continuar.
!pip install -q -U google-genai pypdf python-docx openpyxl pymupdf pillow

In [ ]:
# Código del agente: se descarga/actualiza desde GitHub (siempre la última versión).
# Solo si no hay Internet se usan los .py que estén en /kaggle/input o /kaggle/working.
import glob, os, subprocess, sys
CODE_DIR = "/kaggle/working/agente"
if os.path.isdir(os.path.join(CODE_DIR, ".git")):
    r = subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only", "-q"], capture_output=True, text=True)
else:
    r = subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/gallant-fermat-x83ohv",
                        "https://github.com/mrubilar84-ship-it/PRUEBA-AGENTE_1", CODE_DIR], capture_output=True, text=True)
if r.returncode:
    hits = glob.glob("/kaggle/input/**/doc_review_agent.py", recursive=True)
    if not hits:
        raise RuntimeError("No pude descargar el código (¿Internet: On?) ni hay copia en /kaggle/input:\n" + r.stderr)
    CODE_DIR = os.path.dirname(hits[0])
    print("Sin conexión a GitHub: uso la copia de", CODE_DIR)
sys.path.insert(0, CODE_DIR)
print("Código en:", CODE_DIR)

from doc_review_agent import ReviewConfig, review_folder
import os
from kaggle_secrets import UserSecretsClient
os.environ['GEMINI_API_KEY'] = UserSecretsClient().get_secret('GEMINI_API_KEY')
from gemini_llm import GeminiLLM

In [ ]:
# Modelo por defecto: gemini-3.8-flash (si no existe para tu clave, se elige otro "flash" disponible).
# para más calidad puedes usar uno "pro" (más lento y con cuota gratuita menor): GeminiLLM("<modelo>").
llm = GeminiLLM()
print(llm.check())          # prueba rápida: debe imprimir "listo". Si falla, el mensaje dice por qué.
# print(llm.list_models())  # modelos disponibles para tu clave (si el modelo por defecto no existe)

## Documentos a revisar
Trabajamos sobre una carpeta de trabajo (`/kaggle/working/documentos`) porque los datasets de Kaggle son de solo lectura. Aquí puedes **agregar varios documentos, subirlos desde tu computador, reemplazar o borrar**. Puedes volver a ejecutar esta celda y la de revisión las veces que quieras.

In [ ]:
from doc_manager import DocFolder
docs = DocFolder()                 # carpeta de trabajo: /kaggle/working/documentos
docs.add("documento-1")            # copia los documentos de tu dataset (nombre, URL o carpeta); repite con otros datasets
# docs.add("/kaggle/input/otro/a.pdf", "/kaggle/input/otro/b.docx")     # o archivos sueltos
# docs.upload()                    # botón para subir archivos desde tu computador
# docs.replace("viejo.pdf", "/kaggle/input/documento-1/nuevo.pdf")      # reemplazar uno por otro
# docs.remove("viejo.pdf")         # borrar uno (admite comodines: "*.xlsx")
# docs.clear(confirm=True)         # vaciar la carpeta
docs.list()

In [ ]:
cfg = ReviewConfig(
    plan_mode="skip",   # solo documentos: los planos y las imágenes se omiten
    extra_instructions="Proyecto: <nombre>. Normas aplicables: <ej. AISC 360-16, NCh433>. Unidades: SI.",
)
IN_DIR = docs.folder   # carpeta de trabajo administrada arriba
resultado = review_folder(IN_DIR, "/kaggle/working/informe", cfg, llm=llm)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("/kaggle/working/informe/informe.md", encoding="utf-8").read()))

## Chat con los documentos
Haz preguntas, pide resúmenes o consulta los hallazgos de la revisión. Responde solo con lo que encuentra en los documentos y cita archivo y página.

In [ ]:
from doc_chat import DocChat
chat = DocChat(llm)
chat.add_folder(IN_DIR)
chat.add_review("/kaggle/working/informe/revision.json")   # opcional: permite preguntar por los hallazgos

In [ ]:
print(chat.ask("¿Cuáles son los hallazgos críticos y mayores?"))
print(chat.ask("¿Qué norma se aplica y qué factores de seguridad se usan?"))
print(chat.ask("¿Cuál es el alcance del documento?"))

In [ ]:
# Resumen de todos los documentos. Muestra el avance; con el modelo gratis puede tardar varios minutos.
# Más rápido: resumir uno solo, p. ej. chat.summarize("archivo.pdf"), o usar el botón de la caja de preguntas.
# print(chat.summarize())